In [1]:
import math
import torch

torch.set_printoptions(precision=4, sci_mode=False)

In [2]:
V = torch.tensor([
    [1.0, 0.0],
    [0.0, 2.0],
    [3.0, 1.0],
])
ratios = torch.tensor([0.2, 0.3, 0.5])

pieces = ratios[:, None] * V
print(ratios[:,None])
mixed = pieces.sum(dim=0)

print("每一行乘自己的比例：\n", pieces)
print("三行加起来：", mixed)
print("矩阵乘法得到同样的结果：", ratios @ V)

tensor([[0.2000],
        [0.3000],
        [0.5000]])
每一行乘自己的比例：
 tensor([[0.2000, 0.0000],
        [0.0000, 0.6000],
        [1.5000, 0.5000]])
三行加起来： tensor([1.7000, 1.1000])
矩阵乘法得到同样的结果： tensor([1.7000, 1.1000])


In [3]:
Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0],
])
K = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, -1.0],
])

raw_scores = Q @ K.T
print("Q/K/V的shape：", Q.shape, K.shape, V.shape)
print("K转置的shape：", K.T.shape)
print("相关分数：\n", raw_scores)
print("分数shape：", raw_scores.shape)

Q/K/V的shape： torch.Size([3, 2]) torch.Size([3, 2]) torch.Size([3, 2])
K转置的shape： torch.Size([2, 3])
相关分数：
 tensor([[ 1.,  0.,  1.],
        [ 0.,  1., -1.],
        [ 1.,  1.,  0.]])
分数shape： torch.Size([3, 3])


In [4]:
d = Q.shape[-1]
print(d)
print(math.sqrt(d))
scores = raw_scores / math.sqrt(d)
weights = torch.softmax(scores, dim=-1)

print("缩放后的分数：\n", scores)
print("每行的混合比例：\n", weights)
print("每行比例的总和：", weights.sum(dim=-1))

2
1.4142135623730951
缩放后的分数：
 tensor([[ 0.7071,  0.0000,  0.7071],
        [ 0.0000,  0.7071, -0.7071],
        [ 0.7071,  0.7071,  0.0000]])
每行的混合比例：
 tensor([[0.4011, 0.1978, 0.4011],
        [0.2840, 0.5760, 0.1400],
        [0.4011, 0.4011, 0.1978]])
每行比例的总和： tensor([1., 1., 1.])


In [5]:
output = weights @ V
first_row_check = (
    weights[0, 0] * V[0]
    + weights[0, 1] * V[1]
    + weights[0, 2] * V[2]
)

print("Attention输出：\n", output)
print("输出shape：", output.shape)
print("单独展开第一行：", first_row_check)
print("两种写法一致：", torch.allclose(output[0], first_row_check))

Attention输出：
 tensor([[1.6044, 0.7967],
        [0.7041, 1.2920],
        [0.9944, 1.0000]])
输出shape： torch.Size([3, 2])
单独展开第一行： tensor([1.6044, 0.7967])
两种写法一致： True


In [6]:
Q_changed = Q.clone()
Q_changed[0] = 0.
print(Q_changed)
scores_changed = (Q_changed @ K.T) / math.sqrt(d)
weights_changed = torch.softmax(scores_changed, dim=-1)
output_changed = weights_changed @ V

print("改动前第一行权重：", weights[0])
print("改动后第一行权重：", weights_changed[0])
print("改动后第一行输出：", output_changed[0])
print("其余两行是否保持原样：",
      torch.allclose(output_changed[1:], output[1:]))

tensor([[0., 0.],
        [0., 1.],
        [1., 1.]])
改动前第一行权重： tensor([0.4011, 0.1978, 0.4011])
改动后第一行权重： tensor([0.3333, 0.3333, 0.3333])
改动后第一行输出： tensor([1.3333, 1.0000])
其余两行是否保持原样： True


In [7]:
scores_base = (Q @ K.T) / math.sqrt(Q.shape[-1])
weights_base = torch.softmax(scores_base, dim=-1)
output_base = weights_base @ V

allowed = torch.tensor([
    [True,  False, False],
    [True,  True,  False],
    [True,  True,  True ],
])
print("允许参考的位置：\n", allowed)
print("未屏蔽的权重：\n", weights_base)

允许参考的位置：
 tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])
未屏蔽的权重：
 tensor([[0.4011, 0.1978, 0.4011],
        [0.2840, 0.5760, 0.1400],
        [0.4011, 0.4011, 0.1978]])


In [ ]:
scores_masked = scores_base.masked_fill(~allowed, float("-inf"))
weights_masked = torch.softmax(scores_masked, dim=-1)
print("屏蔽后的分数：\n", scores_masked)
print("屏蔽后的权重：\n", weights_masked)
print("每行权重之和：", weights_masked.sum(dim=-1))

屏蔽后的分数：
 tensor([[0.7071,   -inf,   -inf],
        [0.0000, 0.7071,   -inf],
        [0.7071, 0.7071, 0.0000]])
屏蔽后的权重：
 tensor([[1.0000, 0.0000, 0.0000],
        [0.3302, 0.6698, 0.0000],
        [0.4011, 0.4011, 0.1978]])
每行权重之和： tensor([1., 1., 1.])


In [9]:
output_masked = weights_masked @ V
print("无mask输出：\n", output_base)
print("有mask输出：\n", output_masked)
print("第一行是否等于V第一行：",
      torch.allclose(output_masked[0], V[0]))

无mask输出：
 tensor([[1.6044, 0.7967],
        [0.7041, 1.2920],
        [0.9944, 1.0000]])
有mask输出：
 tensor([[1.0000, 0.0000],
        [0.3302, 1.3395],
        [0.9944, 1.0000]])
第一行是否等于V第一行： True


In [12]:
V_changed = V.clone()
V_changed[2] = torch.tensor([100.,100.])

base_changed = weights_base @ V_changed
masked_changed = weights_masked @ V_changed

print("无mask：前两行原/改：\n", output_base[:2], "\n", base_changed[:2])
print("有mask：前两行原/改：\n", output_masked[:2], "\n", masked_changed[:2])
print("无mask前两行是否保持不变：",
      torch.allclose(output_base[:2], base_changed[:2]))
print("有mask前两行是否保持不变：",
      torch.allclose(output_masked[:2], masked_changed[:2]))
print("有mask最后一行是否保持不变：",
      torch.allclose(output_masked[2],masked_changed[2]))

无mask：前两行原/改：
 tensor([[1.6044, 0.7967],
        [0.7041, 1.2920]]) 
 tensor([[40.5123, 40.5068],
        [14.2869, 15.1549]])
有mask：前两行原/改：
 tensor([[1.0000, 0.0000],
        [0.3302, 1.3395]]) 
 tensor([[1.0000, 0.0000],
        [0.3302, 1.3395]])
无mask前两行是否保持不变： False
有mask前两行是否保持不变： True
有mask最后一行是否保持不变： False
